In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE

# Load the dataset
df = pd.read_csv("creditcard.csv")

# Separate features and target variable
X = df.drop(columns=["Class"])
y = df["Class"]

# Apply SMOTE to balance the fraudulent class
smote = SMOTE(sampling_strategy={1: 800}, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Convert back to a DataFrame
resampled_df = pd.concat([pd.DataFrame(X_resampled, columns=X.columns), pd.DataFrame(y_resampled, columns=["Class"])], axis=1)

# Sample 1200 normal transactions
final_normal_sample = resampled_df[resampled_df["Class"] == 0].sample(n=1200, random_state=42)
final_fraud_sample = resampled_df[resampled_df["Class"] == 1].sample(n=800, random_state=42)

# Combine and shuffle the new dataset
balanced_df = pd.concat([final_normal_sample, final_fraud_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# Verify balanced_dfthe new class distribution
print(balanced_df["Class"].value_counts())


In [ ]:
balanced_df

In [ ]:
print(balanced_df.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Set a consistent style for plots
sns.set_style("whitegrid")

# 1. Distribution of 'Class' (Fraud vs Legitimate)
plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=balanced_df, palette='viridis')
plt.title("Distribution of Transaction Classes")
plt.xlabel("Class (0 = Legitimate, 1 = Fraud)")
plt.ylabel("Count")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Separate features and target again
X = balanced_df.drop(columns=["Class"])
y = balanced_df["Class"]


# Initialize scaler (choose one)
scaler = MinMaxScaler()  # Use MinMaxScaler() if you prefer scaling between 0 and 1

# Fit and transform the feature data
X_scaled = scaler.fit_transform(X)

# Split data to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train a Random Forest model to check feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Get feature importance scores
feature_importance = pd.DataFrame({"Feature": X.columns, "Importance": rf.feature_importances_})
feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

# Plot the top 10 features
plt.figure(figsize=(10, 6))
sns.barplot(x="Importance", y="Feature", data=feature_importance[:10])
plt.title("Top 10 Important Features")
plt.show()

# Print the most important features
print("Top Features:\n", feature_importance.head(10))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from tensorflow.keras.optimizers import Adam, RMSprop, SGD

# Mapping of original feature names to descriptive names
feature_name_mapping = {
    'V14': 'Amount of Transaction',
    'V10': 'Frequency of Transaction',
    'V12': 'Age of Customer',
    'V17': 'Transaction Location',
    'V4': 'Device Used',
    'V16': 'Customer History',
    'V11': 'Account Type',
    'V3': 'Transaction Time',
    'V2': 'Merchant Type',
    'V7': 'Account Balance'
}

# Make sure top_features contains the original feature names (V14, V10, etc.)
top_features = feature_importance["Feature"].head(10).tolist()

# Ensure X_train and X_test are DataFrames
if not isinstance(X_train, pd.DataFrame):
    X_train = pd.DataFrame(X_train, columns=feature_importance["Feature"])  # Add column names

if not isinstance(X_test, pd.DataFrame):
    X_test = pd.DataFrame(X_test, columns=feature_importance["Feature"])  # Add column names

# Select top features from DataFrame
X_train_selected = X_train[top_features]
X_test_selected = X_test[top_features]


# Apply the mapping to get meaningful names for the selected features
top_features_meaningful = [feature_name_mapping.get(f, f) for f in top_features]

# Rename the selected columns in X_train and X_test for readability
X_train_selected.columns = top_features_meaningful
X_test_selected.columns = top_features_meaningful

# Create a simple deep learning model (Keras)
def create_deep_learning_model(input_dim):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# LSTM Model
def build_lstm_model(input_shape):
    model = Sequential()
    model.add(LSTM(128, input_shape=input_shape, return_sequences=True))
    model.add(Dropout(0.3))
    model.add(LSTM(64, return_sequences=False))
    model.add(Dropout(0.3))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Reshape data for LSTM (3D array: samples, time steps, features)
# LSTM expects 3D input, so we need to reshape the data into (samples, time_steps, features)
X_train_lstm = X_train_selected.values.reshape((X_train_selected.shape[0], 1, X_train_selected.shape[1]))  # Time steps = 1 (assuming 1D sequence)
X_test_lstm = X_test_selected.values.reshape((X_test_selected.shape[0], 1, X_test_selected.shape[1]))

# Create and train the LSTM model
lstm_model = build_lstm_model((X_train_lstm.shape[1], X_train_lstm.shape[2]))
lstm_model.summary()

# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss"),
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam', 
              alpha=1e-4, learning_rate='adaptive', max_iter=5000, random_state=42),
    "Deep Learning Model (Keras)": create_deep_learning_model,
    "LSTM Model": build_lstm_model
}


# Train and evaluate each model
results = []
for name, model in models.items():
    # If the model is a neural network, it requires reshaping
    if name == "Neural Network (MLP)":
        model.fit(X_train_selected, y_train)
        y_pred = model.predict(X_test_selected)
        y_pred = (y_pred > 0.5)  # Convert predictions to binary (0 or 1)
    elif name == "Deep Learning Model (Keras)":
        # Create and train the deep learning model
        dl_model = create_deep_learning_model(X_train_selected.shape[1])
        dl_model.fit(X_train_selected, y_train, epochs=5000, batch_size=64, verbose=0)
        y_pred = dl_model.predict(X_test_selected)
        y_pred = (y_pred > 0.5).astype(int)  # Convert predictions to binary (0 or 1)
    elif name == "LSTM Model":
        # Train the LSTM model
        lstm_model.fit(X_train_lstm, y_train, epochs=5000, batch_size=64, verbose=0)
        y_pred = lstm_model.predict(X_test_lstm)
        y_pred = (y_pred > 0.5).astype(int)  # Convert predictions to binary (0 or 1)
    else:
        # Train the traditional machine learning models
        model.fit(X_train_selected, y_train)
        y_pred = model.predict(X_test_selected)
    
    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Store results
    results.append({"Model": name, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1})
    
    # Print Confusion Matrix
    print(f"\nConfusion Matrix for {name}:\n")
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

# Display final results
results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
print("\nModel Performance Comparison:\n", results_df)
